# Step 0-A: 상세 코딩 가이드

> 이 노트북은 **힌트만** 제공합니다. `step_0a_skeleton.ipynb`의 TODO를 직접 채우세요!
>
> 막히면 `solutions/step_0a_solution.ipynb`를 참고하세요.

---
## 1. 쿼리도 좌표계

```
          col 0   1   2   3   4   5   6   7   8
row 0      ·    ·    ·    ·   ②    ·    ·    ·    ·     ← P1 시작
row 1      ·    ·    ·    ·    ·    ·    ·    ·    ·
row 2      ·    ·    ·    ·    ·    ·    ·    ·    ·
row 3      ·    ·    ·    ·    ·    ·    ·    ·    ·
row 4      ·    ·    ·    ·    ·    ·    ·    ·    ·
row 5      ·    ·    ·    ·    ·    ·    ·    ·    ·
row 6      ·    ·    ·    ·    ·    ·    ·    ·    ·
row 7      ·    ·    ·    ·    ·    ·    ·    ·    ·
row 8      ·    ·    ·    ·   ①    ·    ·    ·    ·     ← P0 시작
```

| 플레이어 | 시작 위치 | 이동 방향 | 목표 |
|---|---|---|---|
| Player 0 (①) | `(8, 4)` 맨 아래 중앙 | ↑ row 감소 | row = 0 도달 시 승리 |
| Player 1 (②) | `(0, 4)` 맨 위 중앙 | ↓ row 증가 | row = 8 도달 시 승리 |

⚠️ **row가 증가하면 아래, 감소하면 위!** 수학 좌표계와 반대입니다.  
North(위) = row **감소** = `dr = -1`

---
## 2. 행동(Action) 인코딩 — 총 140개

Q-learning/DQN에서 신경망 출력은 **고정 크기 벡터**여야 합니다.  
모든 가능한 행동을 0~139 번호로 미리 매핑합니다.

### 기본 이동 (액션 0~3)

```
       (r-1, c)
         ↑ 액션 0 (North)
         │
(r, c-1) ← ─ (r, c) ─ → (r, c+1)
  액션 2      현재       액션 3
  (West)     위치       (East)
         │
         ↓ 액션 1 (South)
       (r+1, c)
```

| 액션 | 방향 | (dr, dc) |
|---|---|---|
| 0 | North | `(-1, 0)` |
| 1 | South | `(+1, 0)` |
| 2 | West | `(0, -1)` |
| 3 | East | `(0, +1)` |

### 직선 점프 (액션 4~7)

상대가 바로 옆에 있을 때, 상대를 **뛰어넘어** 2칸 이동합니다.

```
  상대 뒤 (r-2, c) ← 여기로 착지!
         ↑
  상대   (r-1, c)
         ↑ 뛰어넘기
  나     (r, c)

  → 액션 4: Jump North,  (dr, dc) = (-2, 0)
```

| 액션 | 방향 | (dr, dc) | 대응 기본 이동 |
|---|---|---|---|
| 4 | Jump N | `(-2, 0)` | 0 (North) |
| 5 | Jump S | `(+2, 0)` | 1 (South) |
| 6 | Jump W | `(0, -2)` | 2 (West) |
| 7 | Jump E | `(0, +2)` | 3 (East) |

💡 `JUMP_TO_MOVE` 딕셔너리: `{4:0, 5:1, 6:2, 7:3}` — 점프에서 기본방향 찾을 때 사용

### 대각 점프 (액션 8~11)

직선 점프가 **벽이나 보드 끝에 막혔을 때**, 상대 위치에서 좌/우로 빠지는 점프입니다.

```
  ═══════ ← 벽! 직선 점프(3,4) 차단!
  (4,3)  (4,4)  (4,5)
   ↖NW    ②     ↗NE     ← 상대 좌우로 대각 점프
         (5,4)
          ①
```

| 액션 | 방향 | (dr, dc) | 구성 = 기본 방향 2개 조합 |
|---|---|---|---|
| 8 | NW | `(-1, -1)` | **N(0)** + **W(2)** |
| 9 | NE | `(-1, +1)` | **N(0)** + **E(3)** |
| 10 | SW | `(+1, -1)` | **S(1)** + **W(2)** |
| 11 | SE | `(+1, +1)` | **S(1)** + **E(3)** |

💡 `DIAG_COMPONENTS` 딕셔너리: `{8:(0,2), 9:(0,3), 10:(1,2), 11:(1,3)}`  
대각 점프 처리 시, "상대가 N쪽에 있고 직선이 막혔다" → N을 포함하는 대각(NW, NE)을 필터링하는 데 사용

---
## 3. `QuoridorState` 변수 설명

게임 상태를 저장하는 `QuoridorState` 클래스는 다음 6개의 변수를 갖습니다.

1. **`board_size`** (`int`): 보드의 크기. (기본값 9)
2. **`pos`** (`list`): 각 플레이어의 현재 `(row, col)` 좌표 튜플 리스트.
   - 구조: `[ (P0 좌표), (P1 좌표) ]`
   - 초기값: `[ (board_size - 1, board_size // 2), (0, board_size // 2) ]`
3. **`walls_remaining`** (`list`): 각 플레이어가 가진 남은 벽의 개수.
   - 구조: `[ P0의 남은 벽, P1의 남은 벽 ]`
   - 초기값: `[ num_walls, num_walls ]`
4. **`walls_placed`** (`set`): 보드 위에 설치된 벽들의 집합.
   - 구조: `{(row, col, 방향), (row, col, 방향), ...}` (방향 0=가로, 1=세로)
   - 초기값: `set()` (빈 집합. 탐색 속도 최적화를 위해 리스트가 아닌 집합 사용)
5. **`current_player`** (`int`): 현재 차례인 플레이어 번호.
   - 구조: `0` 또는 `1`
   - 초기값: `0` (항상 P0이 먼저 시작)
6. **`goal_rows`** (`list`): 각 플레이어가 도착해야 승리하는 목표 행(row).
   - 구조: `[ P0의 목표 행, P1의 목표 행 ]`
   - 초기값: `[ 0, board_size - 1 ]`

### `copy()` — 왜 깊은 복사가 필요한가

```python
# ❌ 얕은 복사 (위험!)
new.pos = self.pos              # 같은 리스트를 공유!
new.pos[0] = (7, 4)             # 원본도 바뀜!!

# ✅ 깊은 복사
new.pos = list(self.pos)        # 새 리스트 생성
new.walls_placed = set(self.walls_placed)  # 새 set 생성
```

---
## 4. `_is_blocked()` — 벽 차단 확인 ⭐ 가장 중요

인접한 두 셀 사이에 벽이 있는지 판별합니다.

### 벽의 물리적 구조 (한 벽이 2칸 걸침)

```
수평 벽 (r, c, 0)                    수직 벽 (r, c, 1)

row r  : (r,c)    (r,c+1)            row r  : (r,c) │ (r,c+1)
         ══════════════                              ║
row r+1: (r+1,c)  (r+1,c+1)          row r+1: (r+1,c)│(r+1,c+1)
                                                     ║
차단: (r,c)↔(r+1,c)                  차단: (r,c)↔(r,c+1)
      (r,c+1)↔(r+1,c+1)                    (r+1,c)↔(r+1,c+1)
```

### 핵심: "어떤 벽이 이 이동을 막는가?"

**수직 이동 (↑↓)을 막는 것 → 수평 벽 (orientation=0)**

예) `(5,4)` → `(4,4)` 로 북쪽 이동할 때:

```
row 4: ... (4,3)  (4,4)  (4,5) ...
            ═══════════              ← 벽 (4,3,0): col 3~4 걸침 → 차단 ✅
                  ═══════════        ← 벽 (4,4,0): col 4~5 걸침 → 차단 ✅
                        ═══════════  ← 벽 (4,5,0): col 5~6 걸침 → 무관 ❌
       ═══════════                   ← 벽 (4,2,0): col 2~3 걸침 → 무관 ❌
row 5: ... (5,3)  (5,4)  (5,5) ...
                   나↑
```

✅ **규칙**: `(r1,c1)→(r2,c2)` 수직 이동 시  
- `min_r = min(r1, r2)`  
- **확인할 벽 2개**: `(min_r, c, 0)` 과 `(min_r, c-1, 0)`  
- `c-1 >= 0` 경계 체크 필요!

**수평 이동 (←→)을 막는 것 → 수직 벽 (orientation=1)**

예) `(5,4)` → `(5,5)` 로 동쪽 이동할 때:

```
row 4: ... (4,4) │ (4,5) ...
                 ║                   ← 벽 (4,4,1): row 4~5 걸침 → 차단 ✅
row 5: ... (5,4) │ (5,5) ...
                 ║                   ← 벽 (5,4,1): row 5~6 걸침 → 차단 ✅
row 6: ... (6,4) │ (6,5) ...
                 ║                   ← 벽 (6,4,1): row 6~7 걸침 → 무관 ❌
            나 →
```

✅ **규칙**: `(r1,c1)→(r2,c2)` 수평 이동 시  
- `min_c = min(c1, c2)`  
- **확인할 벽 2개**: `(r, min_c, 1)` 과 `(r-1, min_c, 1)`  
- `r-1 >= 0` 경계 체크 필요!

### 의사 코드

```
함수 _is_blocked(r1, c1, r2, c2):
    dr = r2 - r1

    만약 dr ≠ 0 (수직 이동):
        min_r = min(r1, r2)
        c = c1
        만약 (min_r, c, 0) 이 벽 set에 있으면: return True
        만약 c-1 >= 0 이고 (min_r, c-1, 0) 이 벽 set에 있으면: return True

    아니면 (수평 이동):
        min_c = min(c1, c2)
        r = r1
        만약 (r, min_c, 1) 이 벽 set에 있으면: return True
        만약 r-1 >= 0 이고 (r-1, min_c, 1) 이 벽 set에 있으면: return True

    return False
```

⚠️ **흔한 실수**: 수직 이동인데 수직 벽을 확인하는 것!  
**수직 이동 → 수평 벽, 수평 이동 → 수직 벽**  
"가로로 놓인 벽이 세로 이동을 막는다"고 기억하세요.

---
## 5. `_get_legal_moves()` — 합법 이동 ⭐

### 5-1단계: 기본 4방향 이동

```
MOVE_DELTA의 각 (action, (dr, dc))에 대해:
    nr = 현재_row + dr
    nc = 현재_col + dc

    조건 1: (nr, nc)가 보드 안인가?       → _in_bounds
    조건 2: 사이에 벽이 없는가?            → _is_blocked 가 False
    조건 3: 그 칸에 상대가 없는가?         → (nr, nc) ≠ 상대 위치

    세 조건 모두 만족 → legal에 추가
```

### 5-2단계: 직선 점프

상대가 **특정 방향 바로 옆**에 있을 때만 발동합니다.

```
예시: 나(5,4), 상대(4,4)

  (3,4) ← behind (상대 뒤)
  (4,4) ← 상대 ②
  (5,4) ← 나 ①

  Jump N(액션4)의 base 방향 = N(액션0), (dr,dc) = (-1, 0)
  
  1. 상대가 N 방향에 있나? → (5-1, 4) = (4,4) = 상대 위치 ✅
  2. 나↔상대 벽? → _is_blocked(5,4, 4,4) → False ✅
  3. 상대 뒤 = (4-1, 4) = (3,4)
     보드 안? ✅  벽? _is_blocked(4,4, 3,4) → False ✅
  → 직선 점프 가능! 액션 4를 legal에 추가
```

### 5-3단계: 대각 점프 (직선이 막혔을 때)

```
예시: 나(5,4), 상대(4,4), 벽 (3,4,0)

  ═══════ ← 벽! 직선 점프(3,4) 불가
  (4,3)  (4,4)  (4,5)
   ↖NW    ②     ↗NE     ← 상대 좌우로 대각 점프 시도
         (5,4)
          ①

  base 방향 = N(0). 직선 점프 불가 확인됨.

  DIAG_COMPONENTS 순회:
    NW(8) = (0, 2) → base(0)을 포함 ✅ → other = 2(West)
      상대에서 West로: (4, 4-1) = (4,3)
      보드 안? ✅  _is_blocked(4,4, 4,3)? False ✅  나 위치 아님? ✅
      → 액션 8 (NW) legal!

    NE(9) = (0, 3) → base(0)을 포함 ✅ → other = 3(East)
      상대에서 East로: (4, 4+1) = (4,5)
      보드 안? ✅  _is_blocked(4,4, 4,5)? False ✅  나 위치 아님? ✅
      → 액션 9 (NE) legal!

    SW(10) = (1, 2) → base(0)을 포함 안 함 ❌ → 건너뜀
    SE(11) = (1, 3) → base(0)을 포함 안 함 ❌ → 건너뜀
```

### 전체 의사 코드

```
함수 _get_legal_moves():
    p = 현재 플레이어,  opp = 상대
    (pr, pc) = 내 위치,  (opr, opc) = 상대 위치
    legal = 빈 리스트

    ── 1) 기본 4방향 ──
    MOVE_DELTA의 각 (action, (dr, dc))에 대해:
        (nr, nc) = (pr+dr, pc+dc)
        if 보드밖: continue
        if 벽 있음: continue
        if 상대 있음: continue
        legal에 action 추가

    ── 2) 점프 ──
    JUMP_DELTA의 각 jump_action에 대해:
        base = JUMP_TO_MOVE[jump_action]
        (bdr, bdc) = MOVE_DELTA[base]

        if (pr+bdr, pc+bdc) ≠ (opr, opc): continue   # 상대가 이 방향에 없음
        if _is_blocked(pr,pc, opr,opc): continue      # 나↔상대 사이 벽

        behind = (opr+bdr, opc+bdc)

        if behind 보드안 AND 벽 없음:
            legal에 jump_action 추가              # 직선 점프
        else:
            DIAG_COMPONENTS의 각 (diag, (c1, c2))에 대해:
                if base가 c1도 c2도 아니면: continue
                other = c1이 base면 c2, 아니면 c1
                (odr, odc) = MOVE_DELTA[other]
                side = (opr+odr, opc+odc)
                if side 보드안 AND 벽 없음 AND side ≠ 내 위치:
                    legal에 diag 추가              # 대각 점프

    return legal
```

---
## 6. `step()` 구현 힌트

```
함수 step(action):
    p = 현재 플레이어

    if action < 12 (이동/점프):
        action이 MOVE_DELTA에 있으면 → (dr,dc) = MOVE_DELTA[action]
        action이 JUMP_DELTA에 있으면 → (dr,dc) = JUMP_DELTA[action]
        action이 DIAG_DELTA에 있으면 → (dr,dc) = DIAG_DELTA[action]

        pos[p] = (pos[p][0] + dr, pos[p][1] + dc)
    else:
        raise NotImplementedError   # 벽 → Step 0-B

    done = _check_win(p)
    reward = 1.0 if done else 0.0

    if not done:                    # ⚠️ 승리 시 턴 전환 안 함!
        current_player = 1 - p

    return (state, reward, done)
```

---
## 7. `_check_win()` 구현 힌트

가장 간단한 함수입니다:

```
return state.pos[player]의 row == state.goal_rows[player]
```

`state.pos[player]`는 `(row, col)` 튜플이므로 `[0]`이 row입니다.

---
## 8. `render()` (선택사항)

디버깅에 유용하지만 필수는 아닙니다.

**최소 버전:**
```
각 row에 대해:
    각 col에 대해:
        if (r,c) == P0: "①" 출력
        elif (r,c) == P1: "②" 출력
        else: "·" 출력
    줄바꿈
```

정답의 render()를 복사해도 OK!

---
## ⚠️ 자주 하는 실수 모음

| 실수 | 증상 | 해결 |
|---|---|---|
| 수직 이동인데 수직 벽 확인 | 벽이 있는데 통과됨 | **수직 이동 → 수평 벽**, 수평 이동 → 수직 벽 |
| 벽 2칸 걸침 미처리 | 일부 벽만 인식 | c와 **c-1** 두 곳 모두 확인 |
| c-1 경계 체크 누락 | 잘못된 결과 | `c - 1 >= 0` 조건 추가 |
| `copy()`에서 얕은 복사 | 한 상태 변경이 다른 데 영향 | `list()`, `set()`으로 감싸기 |
| 승리 시 턴 전환 | 누가 이겼는지 모름 | `if not done:` 안에서만 전환 |
| 상대가 있는 칸으로 기본 이동 | 겹침 발생 | 3번째 조건으로 걸러야 함 |